In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sohansakib75/yuyvvty")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/sohansakib75/yuyvvty


In [4]:
import cv2
import numpy as np
import os
from PIL import Image
from tqdm import tqdm

base_dir = "/kaggle/input/datasets/sohansakib75/yuyvvty/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation"
classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]
TARGET_SIZE = (224, 224)

output_base = "/kaggle/working/preprocessed_all"

eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def alpha_trimmed_mean_filter(img, kernel_size=3, alpha=2):
    pad = kernel_size // 2
    padded = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    filtered = np.zeros_like(img)
    for i in range(pad, padded.shape[0] - pad):
        for j in range(pad, padded.shape[1] - pad):
            kernel = padded[i - pad:i + pad + 1, j - pad:j + pad + 1].flatten()
            kernel.sort()
            trimmed = kernel[alpha:-alpha]
            filtered[i - pad, j - pad] = np.mean(trimmed)
    return filtered

def zoom_roi_with_padding(img, bbox, zoom_factor=1.3):
    h, w = img.shape[:2]
    x, y, bw, bh = bbox
    
    # Calculate ROI center
    cx = x + bw // 2
    cy = y + bh // 2
    
    # Calculate new ROI size
    new_w = int(bw * zoom_factor)
    new_h = int(bh * zoom_factor)
    
    # Calculate new bounding box with padding around ROI center
    x1 = max(cx - new_w // 2, 0)
    y1 = max(cy - new_h // 2, 0)
    x2 = min(cx + new_w // 2, w)
    y2 = min(cy + new_h // 2, h)
    
    # Crop and zoom ROI area
    roi = img[y1:y2, x1:x2]
    roi_resized = cv2.resize(roi, (w, h), interpolation=cv2.INTER_LINEAR)
    
    return roi_resized

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    original_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize full image first
    resized = cv2.resize(original_rgb, TARGET_SIZE)

    # Convert to grayscale for detection
    gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)

    # Detect eyes
    eyes = eye_cascade.detectMultiScale(gray, 1.3, 5)
    if len(eyes) > 0:
        bbox = eyes[0]
    else:
        # Detect face
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        if len(faces) > 0:
            bbox = faces[0]
        else:
            # If no detection, use whole image
            bbox = (0, 0, resized.shape[1], resized.shape[0])

    # Zoom ROI with padding, keep image size
    zoomed_img = zoom_roi_with_padding(resized, bbox, zoom_factor=1.3)

    # Apply CLAHE on grayscale
    gray_zoomed = cv2.cvtColor(zoomed_img, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_img = clahe.apply(gray_zoomed)

    # Apply alpha trimmed mean filter
    filtered_img = alpha_trimmed_mean_filter(clahe_img)

    # Convert back to RGB
    processed_img = cv2.merge([filtered_img]*3)

    return processed_img

# Process all images, save results
for cls in classes:
    class_input_dir = os.path.join(base_dir, cls)
    class_output_dir = os.path.join(output_base, cls)
    os.makedirs(class_output_dir, exist_ok=True)

    image_files = [f for f in os.listdir(class_input_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for i, img_file in enumerate(tqdm(image_files, desc=f"Processing {cls}")):
        img_path = os.path.join(class_input_dir, img_file)
        preprocessed_img = preprocess_image(img_path)
        if preprocessed_img is not None:
            save_path = os.path.join(class_output_dir, f"image_{i+1}.jpg")
            Image.fromarray(preprocessed_img).save(save_path)

print("All images preprocessed and saved successfully!")


Processing Conjunctivitis: 100%|██████████| 357/357 [02:33<00:00,  2.32it/s]

All images preprocessed and saved successfully!


In [11]:
import os
import random
import shutil
from collections import defaultdict

# ============================================================
# SOURCE DATASET
# ============================================================

augmented_dir = "/kaggle/working/augmented_all"

classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]

split_ratios = {'train': 0.8, 'val': 0.1, 'test': 0.1}

output_dir = "/kaggle/working"

splits = ["train", "val", "test"]

# ============================================================
# CREATE FOLDERS
# ============================================================

for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

print("✔ Output folders created")

# ============================================================
# SPLIT DATA
# ============================================================

for cls in classes:
    class_dir = os.path.join(augmented_dir, cls)

    if not os.path.exists(class_dir):
        print(f"❌ Missing class folder: {class_dir}")
        continue

    image_files = [
        os.path.join(class_dir, f)
        for f in os.listdir(class_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    random.shuffle(image_files)

    n = len(image_files)
    n_train = int(split_ratios['train'] * n)
    n_val = int(split_ratios['val'] * n)

    train_files = image_files[:n_train]
    val_files = image_files[n_train:n_train + n_val]
    test_files = image_files[n_train + n_val:]

    # Copy files
    for f in train_files:
        shutil.copy(f, os.path.join(output_dir, "train", cls))

    for f in val_files:
        shutil.copy(f, os.path.join(output_dir, "val", cls))

    for f in test_files:
        shutil.copy(f, os.path.join(output_dir, "test", cls))

print("\n✔ Dataset split completed")

# ============================================================
# CHECK SPLIT STRUCTURE
# ============================================================

print("\n🔍 Checking split results...\n")

for split in splits:
    print(f"📁 {split.upper()}")

    split_path = os.path.join(output_dir, split)

    for cls in classes:
        class_path = os.path.join(split_path, cls)

        if not os.path.exists(class_path):
            print(f"   ❌ Missing {cls}")
            continue

        n_imgs = len([
            f for f in os.listdir(class_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

        print(f"   ✔ {cls:<15}: {n_imgs} images")

print("\n✅ Done")

✔ Output folders created

✔ Dataset split completed

🔍 Checking split results...

📁 TRAIN
   ✔ Eyelid         : 1680 images
   ✔ Normal         : 2076 images
   ✔ Cataract       : 1740 images
   ✔ Uveitis        : 1248 images
   ✔ Conjunctivitis : 1142 images
📁 VAL
   ✔ Eyelid         : 210 images
   ✔ Normal         : 259 images
   ✔ Cataract       : 217 images
   ✔ Uveitis        : 156 images
   ✔ Conjunctivitis : 142 images
📁 TEST
   ✔ Eyelid         : 210 images
   ✔ Normal         : 261 images
   ✔ Cataract       : 219 images
   ✔ Uveitis        : 157 images
   ✔ Conjunctivitis : 144 images

✅ Done


In [13]:
import os
import hashlib
import imagehash
from PIL import Image
import pandas as pd
from tabulate import tabulate

# ============================================================
# PATHS
# ============================================================

base_dir = "/kaggle/working"

splits = {
    "train": os.path.join(base_dir, "train"),
    "val": os.path.join(base_dir, "val"),
    "test": os.path.join(base_dir, "test")
}

# ============================================================
# LOAD IMAGES
# ============================================================

def get_images(folder):
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                paths.append(os.path.join(root, f))
    return paths

# ============================================================
# HASH FUNCTIONS
# ============================================================

def md5_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def phash(path):
    try:
        img = Image.open(path).convert("RGB")
        return imagehash.phash(img)
    except:
        return None

# ============================================================
# BUILD DATA
# ============================================================

hash_data = {}
image_counts = {}

for split in splits:
    images = get_images(splits[split])
    image_counts[split] = len(images)

    md5_map = {}
    phash_map = {}

    for img in images:
        md5_map[img] = md5_hash(img)
        ph = phash(img)
        if ph is not None:
            phash_map[img] = ph

    hash_data[split] = {
        "md5": md5_map,
        "phash": phash_map
    }

# ============================================================
# FUNCTIONS
# ============================================================

def exact_overlap(a, b):
    set_a = set(hash_data[a]["md5"].values())
    set_b = set(hash_data[b]["md5"].values())
    return len(set_a & set_b)

def near_overlap(a, b, threshold=5):
    count = 0
    for h1 in hash_data[a]["phash"].values():
        for h2 in hash_data[b]["phash"].values():
            if (h1 - h2) <= threshold:
                count += 1
    return count

# ============================================================
# PAIRWISE ANALYSIS
# ============================================================

pairs = [
    ("train", "val"),
    ("train", "test"),
    ("val", "test")
]

results = []

for a, b in pairs:

    exact = exact_overlap(a, b)
    near = near_overlap(a, b, threshold=5)

    total_a = image_counts[a]
    total_b = image_counts[b]

    exact_pct = (exact / min(total_a, total_b)) * 100 if min(total_a, total_b) > 0 else 0
    near_pct = (near / min(total_a, total_b)) * 100 if min(total_a, total_b) > 0 else 0

    results.append({
        "Comparison": f"{a.upper()} vs {b.upper()}",
        "Images A": total_a,
        "Images B": total_b,
        "Exact Dup": exact,
        "Exact %": round(exact_pct, 3),
        "Near Dup": near,
        "Near %": round(near_pct, 3)
    })

# ============================================================
# SELF CHECK (within split duplicates)
# ============================================================

def self_duplicate_rate(split):
    hashes = list(hash_data[split]["md5"].values())
    return len(hashes) - len(set(hashes))

self_results = []

for s in splits:
    dup = self_duplicate_rate(s)
    total = image_counts[s]
    pct = (dup / total) * 100 if total > 0 else 0

    self_results.append({
        "Split": s,
        "Total Images": total,
        "Self Duplicates": dup,
        "Self Dup %": round(pct, 3)
    })

# ============================================================
# OUTPUT TABLES
# ============================================================

df1 = pd.DataFrame(results)
df2 = pd.DataFrame(self_results)

print("\n🔍 CROSS-SPLIT LEAKAGE REPORT\n")
print(tabulate(df1, headers="keys", tablefmt="grid", showindex=False))

print("\n🔍 WITHIN-SPLIT DUPLICATION REPORT\n")
print(tabulate(df2, headers="keys", tablefmt="grid", showindex=False))


🔍 CROSS-SPLIT LEAKAGE REPORT

+---------------+------------+------------+-------------+-----------+------------+----------+
| Comparison    |   Images A |   Images B |   Exact Dup |   Exact % |   Near Dup |   Near % |
+===============+============+============+=============+===========+============+==========+
| TRAIN vs VAL  |       7886 |        984 |           0 |         0 |        187 |   19.004 |
+---------------+------------+------------+-------------+-----------+------------+----------+
| TRAIN vs TEST |       7886 |        991 |           0 |         0 |        191 |   19.273 |
+---------------+------------+------------+-------------+-----------+------------+----------+
| VAL vs TEST   |        984 |        991 |           0 |         0 |         26 |    2.642 |
+---------------+------------+------------+-------------+-----------+------------+----------+

🔍 WITHIN-SPLIT DUPLICATION REPORT

+---------+----------------+-------------------+--------------+
| Split   |   Total Ima